# Data Quality Check (Features + Groups)

This notebook validates two key checks before running GroupKFold training:
1. **Group column has no NaNs** (otherwise `GroupKFold` fails).
2. **The correct group column is used** (exists in the data and is not treated as a feature).

It also reports basic statistics for **input features** (not all metadata) and target values.

**Inputs are pulled from** `deep_learning/config.py` and **data is loaded via** `deep_learning/utils.py`.


In [3]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

# Local modules (same directory as this notebook)
import config
import utils

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


/Users/legallyoverworked/opt/anaconda3/envs/adventml_env/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/legallyoverworked/opt/anaconda3/envs/adventml_env/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/Users/legallyoverworked/opt/anaconda3/envs/adventml_env/lib/python3.10/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime ve

In [4]:
# Resolve data paths from config (override with env var if needed)
CONFIG_DIR = Path(config.__file__).resolve().parent

DATA_DIR = Path(os.environ.get('OGT_DATA_DIR', config.DATA_PATH)).expanduser()
if not DATA_DIR.is_absolute():
    DATA_DIR = (CONFIG_DIR / DATA_DIR).resolve()

TRAIN_PATH = DATA_DIR / config.TRAINING_FILE
TEST_PATH = DATA_DIR / config.TESTING_FILE
WEIGHTS_PATH = DATA_DIR / config.WEIGHTS_FILE

print(f'CONFIG_DIR: {CONFIG_DIR}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'TRAIN_PATH: {TRAIN_PATH} (exists={TRAIN_PATH.exists()})')
print(f'TEST_PATH: {TEST_PATH} (exists={TEST_PATH.exists()})')
print(f'WEIGHTS_PATH: {WEIGHTS_PATH} (exists={WEIGHTS_PATH.exists()})')

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f'Training file not found at {TRAIN_PATH}. '
        'Set OGT_DATA_DIR or update config.DATA_PATH.'
    )


CONFIG_DIR: /Volumes/NUWA/projects/ogtfinder_expansion/dl_ogt/deep_learning
DATA_DIR: /Volumes/NUWA/projects/ogtfinder_expansion/dl_ogt/data
TRAIN_PATH: /Volumes/NUWA/projects/ogtfinder_expansion/dl_ogt/data/train_genus.csv (exists=True)
TEST_PATH: /Volumes/NUWA/projects/ogtfinder_expansion/dl_ogt/data/test_genus.csv (exists=True)
WEIGHTS_PATH: /Volumes/NUWA/projects/ogtfinder_expansion/dl_ogt/data/weights_train_genus.json (exists=True)


In [5]:
# Load data using utils
df = utils.load_data(str(TRAIN_PATH))
print(f'Data shape: {df.shape}')
df.head(3)


Data shape: (5715, 75)


,ncbiTaxID_new,species_id,species,superkingdom_id,superkingdom,phylum_id,phylum,class_id,class,order_id,order,family_id,family,genus_id,genus,median_temp,superkingdom_dummy,B1_mean,B2_mean,B3_mean,B4_mean,B5_mean,B6_mean,B7_mean,B8_mean,B9_mean,B10_mean,PP1_mean,PP2_mean,PP3_mean,F1_mean,F2_mean,F3_mean,F4_mean,F5_mean,F6_mean,K1_mean,K2_mean,K3_mean,K4_mean,K5_mean,K6_mean,K7_mean,K8_mean,K9_mean,K10_mean,MSWHIM1_mean,MSWHIM2_mean,MSWHIM3_mean,ST1_mean,ST2_mean,ST3_mean,ST4_mean,ST5_mean,ST6_mean,ST7_mean,ST8_mean,T1_mean,T2_mean,T3_mean,T4_mean,T5_mean,VHSE1_mean,VHSE2_mean,VHSE3_mean,VHSE4_mean,VHSE5_mean,VHSE6_mean,VHSE7_mean,VHSE8_mean,Z1_mean,Z2_mean,Z3_mean,Z4_mean,Z5_mean
0,1000565,378211,Methyloversatilis universalis,2,Bacteria,1224.0,Pseudomonadota,28216.0,Betaproteobacteria,32003.0,Nitrosomonadales,2008793.0,Sterolibacteriaceae,378210.0,Methyloversatilis,33.5,0.0,0.110224,-0.280964,-0.086016,-0.039235,-0.048684,0.139295,0.121483,-0.003743,0.169911,0.038644,-0.210741,-0.326850,0.041037,-0.192825,0.063184,-0.181846,0.491155,0.105582,-0.046621,-0.167073,-0.317491,-0.029819,0.038133,-0.229969,-0.328622,0.033278,-0.031340,-0.022262,0.080792,-0.352821,0.372638,-0.379959,-0.907794,-0.263615,-0.224595,-0.008501,-0.158798,-0.441609,0.152590,0.324133,-4.875221,-0.120255,0.016974,0.295248,0.766894,-0.008371,-0.170978,-0.266584,-0.127238,-0.072206,-0.302647,0.225350,-0.001783,0.050552,-0.658494,-0.295796,-0.342652,0.318035
1,1001240,1001240,Cryobacterium roopkundense,2,Bacteria,201174.0,Actinomycetota,1760.0,Actinomycetes,85006.0,Micrococcales,85023.0,Microbacteriaceae,69578.0,Cryobacterium,16.5,0.0,0.092738,-0.314407,-0.061852,-0.023578,-0.085441,0.159297,0.118142,-0.045568,0.178176,0.053113,-0.227196,-0.353807,0.041366,-0.194377,0.025165,-0.208324,0.530355,0.101129,-0.055975,-0.145225,-0.371543,0.022591,0.021261,-0.231975,-0.343153,-0.020327,-0.039230,-0.053557,0.067232,-0.377403,0.384867,-0.417149,-0.935075,-0.305124,-0.259211,-0.004569,-0.150043,-0.491695,0.131638,0.306762,-5.034624,-0.211547,0.075812,0.314988,0.807070,0.010752,-0.215698,-0.314119,-0.186243,-0.095203,-0.351779,0.197998,-0.001541,0.006471,-0.783053,-0.291425,-0.423549,0.325088
2,100176,100176,Papillibacter cinnamivorans,2,Bacteria,1239.0,Bacillota,186801.0,Clostridia,186802.0,Eubacteriales,216572.0,Oscillospiraceae,100175.0,Papillibacter,37.0,0.0,0.094834,-0.258033,-0.130054,-0.008035,-0.052934,0.164709,0.067194,-0.014721,0.143548,-0.001008,-0.163279,-0.315016,0.030422,-0.187215,0.056277,-0.131496,0.451254,0.042646,-0.034884,-0.133868,-0.253790,0.026708,0.065717,-0.161404,-0.323284,0.037984,0.037794,-0.037556,0.041627,-0.342556,0.373550,-0.362632,-0.879654,-0.259016,-0.194851,-0.014430,-0.132282,-0.317824,0.106179,0.263695,-4.711838,-0.010895,-0.014165,0.283903,0.690231,-0.014914,-0.137367,-0.212068,-0.114149,-0.080273,-0.275353,0.206342,-0.048442,0.031183,-0.611221,-0.314804,-0.439366,0.230955


In [6]:
# Feature/target/group columns from config
feature_cols = list(config.FEATURE_COLUMNS)
target_col = config.TARGET
group_col = config.GROUP_COLUMN

print(f'Num configured features: {len(feature_cols)}')
print(f'Target column: {target_col}')
print(f'Group column: {group_col}')


Num configured features: 58
Target column: median_temp
Group column: genus_id


In [7]:
# Check feature columns exist (use config-defined feature list, not metadata)
missing_features = [c for c in feature_cols if c not in df.columns]
extra_features = [c for c in df.columns if c in feature_cols]

if missing_features:
    print('Missing feature columns:')
    print(missing_features)
else:
    print('All configured feature columns are present.')

# Ensure group column is NOT in feature list
if group_col in feature_cols:
    print(f'ERROR: group column `{group_col}` is included in FEATURE_COLUMNS.')
else:
    print('Group column is not included in feature columns (OK).')


All configured feature columns are present.
Group column is not included in feature columns (OK).


In [8]:
# Check group column availability and NaNs (Check #1 and #2)
if group_col not in df.columns:
    raise ValueError(f'Group column `{group_col}` not found in data!')

group_series = df[group_col]
num_group_nans = int(group_series.isna().sum())
num_groups = int(group_series.nunique(dropna=True))

print(f'Group column dtype: {group_series.dtype}')
print(f'Unique groups (non-NaN): {num_groups}')
print(f'NaNs in group column: {num_group_nans}')

if num_group_nans > 0:
    print('Sample rows with NaN group values:')
    display(df.loc[group_series.isna(), [group_col, target_col] + feature_cols].head(5))


Group column dtype: float64
Unique groups (non-NaN): 1793
NaNs in group column: 0


In [9]:
# Check target column presence and NaNs
if target_col not in df.columns:
    raise ValueError(f'Target column `{target_col}` not found in data!')

target_series = df[target_col]
print(f'Target dtype: {target_series.dtype}')
print(f'NaNs in target column: {int(target_series.isna().sum())}')
target_series.describe()


Target dtype: float64
NaNs in target column: 0


count    5715.000000
mean       33.354061
std        11.633247
min         4.000000
25%        28.000000
50%        30.000000
75%        37.000000
max       106.000000
Name: median_temp, dtype: float64

In [10]:
# Basic feature stats (only configured input features)
features_df = df[feature_cols]

# Non-numeric features check
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(features_df[c])]
if non_numeric:
    print('Non-numeric feature columns:')
    print(non_numeric)
else:
    print('All feature columns are numeric (OK).')

# Missing values per feature
missing_per_feature = features_df.isna().sum().sort_values(ascending=False)
print('Top 10 features with missing values:')
display(missing_per_feature.head(10))

# Summary stats
features_df.describe().T.head(20)


All feature columns are numeric (OK).
Top 10 features with missing values:


B1_mean         0
T4_mean         0
MSWHIM3_mean    0
ST1_mean        0
ST2_mean        0
ST3_mean        0
ST4_mean        0
ST5_mean        0
ST6_mean        0
ST7_mean        0
dtype: int64

,count,mean,std,min,25%,50%,75%,max
B1_mean,5715.0,0.106872,0.025870,0.003526,0.090265,0.103543,0.120889,0.206461
B2_mean,5715.0,-0.250950,0.036603,-0.340222,-0.280243,-0.250133,-0.221383,-0.172991
B3_mean,5715.0,-0.134763,0.064217,-0.292296,-0.193476,-0.137179,-0.076463,0.002692
B4_mean,5715.0,-0.014687,0.028459,-0.118421,-0.037268,-0.017945,0.006165,0.080446
B5_mean,5715.0,-0.066814,0.016793,-0.135213,-0.075162,-0.065661,-0.055646,0.015287
B6_mean,5715.0,0.154366,0.017011,0.100632,0.142382,0.152705,0.166046,0.213790
B7_mean,5715.0,0.068996,0.050493,-0.033011,0.023189,0.066829,0.115676,0.189808
B8_mean,5715.0,-0.022957,0.017645,-0.141508,-0.033775,-0.022629,-0.011307,0.048075
B9_mean,5715.0,0.142409,0.032745,0.039674,0.115859,0.141608,0.171832,0.210965
B10_mean,5715.0,0.015498,0.034776,-0.099938,-0.009474,0.024548,0.042423,0.082559


## Notes
- If **NaNs are present in the group column**, GroupKFold will fail. Fix by dropping/imputing those rows or correcting the group assignment.
- If any **configured feature columns are missing**, update the data or adjust `FEATURE_COLUMNS` in `config.py`.
- This notebook intentionally limits checks to the **configured input features**, not all metadata columns.
